# Fraud Detection with Snowflake ML

End-to-end fraud detection on transaction data (~1.75M rows) using **XGBoost** and Snowflake ML capabilities:

- **Feature Store** — Register customer and terminal entities with rolling-window feature views
- **Experiment Tracking** — Compare a baseline model against a feature-enriched model
- **Model Registry** — Version, log, and promote the best-performing classifier

The pipeline runs two experiments:
1. **Baseline** — 5 basic time features (hour, day of week, weekend/night flags)
2. **Feature Store** — 17 features including customer velocity, terminal risk, and spending patterns

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "ML_DEMO"
SCHEMA = "FRAUD_DETECTION"

session.sql(f"CREATE OR REPLACE DATABASE {DATABASE}").collect()
session.sql(f"CREATE OR REPLACE SCHEMA {DATABASE}.{SCHEMA}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Using {DATABASE}.{SCHEMA}")

## Data Ingestion

Load raw transaction data from CSV and create a Snowflake table.

In [ ]:
df = pd.read_csv('data/transactions.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFraud class distribution:")
print(df['TX_FRAUD'].value_counts())
print(f"\nFraud rate: {df['TX_FRAUD'].mean():.4%}")

session.write_pandas(df, 'TRANSACTIONS_RAW', auto_create_table=True, overwrite=True)
print(f"\nTable TRANSACTIONS_RAW created with {len(df)} records")

## Feature Store Setup

Initialize the Feature Store and register customer/terminal entities.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
from snowflake.snowpark import Window
from snowflake.snowpark.functions import (
    hour, dayofweek, when, col, count, sum, avg, stddev,
    make_interval, to_timestamp
)

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="Customer entity for fraud detection features"
)

terminal_entity = Entity(
    name="TERMINAL",
    join_keys=["TERMINAL_ID"],
    desc="Terminal entity for fraud detection features"
)

try:
    fs.register_entity(customer_entity)
    print("Customer entity registered")
except:
    print("Customer entity already exists")

try:
    fs.register_entity(terminal_entity)
    print("Terminal entity registered")
except:
    print("Terminal entity already exists")

fs.list_entities().show()

In [ ]:
# Customer Feature View
df_raw = session.table('TRANSACTIONS_RAW')
df_raw = df_raw.with_column('TX_DATETIME', to_timestamp('TX_DATETIME'))

customer_window_1h = Window.partition_by('CUSTOMER_ID').order_by('TX_DATETIME').range_between(
    -make_interval(hours=1), Window.currentRow
)
customer_window_24h = Window.partition_by('CUSTOMER_ID').order_by('TX_DATETIME').range_between(
    -make_interval(hours=24), Window.currentRow
)
customer_window_7d = Window.partition_by('CUSTOMER_ID').order_by('TX_DATETIME').range_between(
    -make_interval(days=7), Window.currentRow
)

customer_features_df = df_raw.select(
    col('CUSTOMER_ID'),
    col('TX_DATETIME'),
    count('TRANSACTION_ID').over(customer_window_1h).alias('CUST_TX_COUNT_1H'),
    count('TRANSACTION_ID').over(customer_window_24h).alias('CUST_TX_COUNT_24H'),
    count('TRANSACTION_ID').over(customer_window_7d).alias('CUST_TX_COUNT_7D'),
    sum('TX_AMOUNT').over(customer_window_1h).alias('CUST_AMOUNT_1H'),
    sum('TX_AMOUNT').over(customer_window_24h).alias('CUST_AMOUNT_24H'),
    avg('TX_AMOUNT').over(customer_window_7d).alias('CUST_AVG_AMOUNT_7D'),
    stddev('TX_AMOUNT').over(customer_window_7d).alias('CUST_STD_AMOUNT_7D')
)

customer_fv = FeatureView(
    name="CUSTOMER_FRAUD_FEATURES",
    entities=[customer_entity],
    feature_df=customer_features_df,
    timestamp_col="TX_DATETIME",
    refresh_freq="1 hour",
    desc="Customer behavioral features for fraud detection"
)

customer_fv = customer_fv.attach_feature_desc({
    "CUST_TX_COUNT_1H": "Number of transactions by customer in last hour",
    "CUST_TX_COUNT_24H": "Number of transactions by customer in last 24 hours",
    "CUST_TX_COUNT_7D": "Number of transactions by customer in last 7 days",
    "CUST_AMOUNT_1H": "Total amount spent by customer in last hour",
    "CUST_AMOUNT_24H": "Total amount spent by customer in last 24 hours",
    "CUST_AVG_AMOUNT_7D": "Average transaction amount for customer over 7 days",
    "CUST_STD_AMOUNT_7D": "Std deviation of transaction amounts for customer over 7 days"
})

try:
    registered_customer_fv = fs.register_feature_view(feature_view=customer_fv, version="v1", block=True)
    print("Customer feature view registered")
except:
    registered_customer_fv = fs.get_feature_view("CUSTOMER_FRAUD_FEATURES", "v1")
    print("Customer feature view already exists, retrieved existing")

In [ ]:
# Terminal Feature View
terminal_window_1h = Window.partition_by('TERMINAL_ID').order_by('TX_DATETIME').range_between(
    -make_interval(hours=1), Window.currentRow
)
terminal_window_24h = Window.partition_by('TERMINAL_ID').order_by('TX_DATETIME').range_between(
    -make_interval(hours=24), Window.currentRow
)
terminal_window_7d = Window.partition_by('TERMINAL_ID').order_by('TX_DATETIME').range_between(
    -make_interval(days=7), Window.currentRow
)

terminal_features_df = df_raw.select(
    col('TERMINAL_ID'),
    col('TX_DATETIME'),
    count('TRANSACTION_ID').over(terminal_window_1h).alias('TERM_TX_COUNT_1H'),
    count('TRANSACTION_ID').over(terminal_window_24h).alias('TERM_TX_COUNT_24H'),
    sum('TX_AMOUNT').over(terminal_window_24h).alias('TERM_AMOUNT_24H'),
    avg('TX_AMOUNT').over(terminal_window_7d).alias('TERM_AVG_AMOUNT_7D'),
    sum(when(col('TX_FRAUD') == 1, 1).otherwise(0)).over(terminal_window_7d).alias('TERM_FRAUD_COUNT_7D')
)

terminal_fv = FeatureView(
    name="TERMINAL_FRAUD_FEATURES",
    entities=[terminal_entity],
    feature_df=terminal_features_df,
    timestamp_col="TX_DATETIME",
    refresh_freq="1 hour",
    desc="Terminal behavioral features for fraud detection"
)

terminal_fv = terminal_fv.attach_feature_desc({
    "TERM_TX_COUNT_1H": "Number of transactions at terminal in last hour",
    "TERM_TX_COUNT_24H": "Number of transactions at terminal in last 24 hours",
    "TERM_AMOUNT_24H": "Total amount processed at terminal in last 24 hours",
    "TERM_AVG_AMOUNT_7D": "Average transaction amount at terminal over 7 days",
    "TERM_FRAUD_COUNT_7D": "Number of fraudulent transactions at terminal in last 7 days"
})

try:
    registered_terminal_fv = fs.register_feature_view(feature_view=terminal_fv, version="v1", block=True)
    print("Terminal feature view registered")
except:
    registered_terminal_fv = fs.get_feature_view("TERMINAL_FRAUD_FEATURES", "v1")
    print("Terminal feature view already exists, retrieved existing")

## Experiment Tracking Setup

Initialize experiment tracking and model registry.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.registry import Registry
from snowflake.ml.model import task
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

exp = ExperimentTracking(session=session)
exp.set_experiment("FRAUD_DETECTION_EXPERIMENT")

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

print("Experiment tracking initialized")
print("Model registry connected")

## Experiment 1: Baseline Model (Basic Features)

XGBoost classifier with only basic time-based features — no Feature Store enrichment.

In [ ]:
df_basic = session.table('TRANSACTIONS_RAW')
df_basic = df_basic.with_column('TX_DATETIME', to_timestamp('TX_DATETIME'))
df_basic = df_basic.with_columns(
    ['HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT'],
    [
        hour('TX_DATETIME'),
        dayofweek('TX_DATETIME'),
        when(dayofweek('TX_DATETIME') >= 6, True).otherwise(False),
        when(hour('TX_DATETIME').between(0, 6), True).otherwise(False)
    ]
)

df_pandas = df_basic.to_pandas()

basic_features = ['TX_AMOUNT', 'HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT']
X_basic = df_pandas[basic_features].astype(float)
y = df_pandas['TX_FRAUD'].astype(int)

X_train_basic, X_test_basic, y_train, y_test = train_test_split(
    X_basic, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

print(f"Training set: {X_train_basic.shape[0]} samples")
print(f"Test set: {X_test_basic.shape[0]} samples")
print(f"Features: {basic_features}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

model_baseline = XGBClassifier(
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=model_baseline,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

with exp.start_run("baseline_basic_features"):
    exp.log_params({
        "model_type": "XGBClassifier",
        "feature_set": "basic",
        "n_features": len(basic_features),
        "n_train_rows": len(X_train_basic),
        "n_test_rows": len(X_test_basic),
        "param_grid": str(param_grid),
        "scale_pos_weight": scale_pos_weight
    })

    grid_search.fit(X_train_basic, y_train)
    best_model = grid_search.best_estimator_

    exp.log_params({
        "best_n_estimators": grid_search.best_params_['n_estimators'],
        "best_max_depth": grid_search.best_params_['max_depth'],
        "best_learning_rate": grid_search.best_params_['learning_rate'],
        "best_cv_score": grid_search.best_score_
    })

    y_pred_baseline = best_model.predict(X_test_basic)
    y_proba_baseline = best_model.predict_proba(X_test_basic)[:, 1]

    metrics_baseline = {
        "accuracy": float(accuracy_score(y_test, y_pred_baseline)),
        "precision": float(precision_score(y_test, y_pred_baseline, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred_baseline, zero_division=0)),
        "f1_score": float(f1_score(y_test, y_pred_baseline, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_proba_baseline)),
        "pr_auc": float(average_precision_score(y_test, y_proba_baseline))
    }
    exp.log_metrics(metrics_baseline)

    mv_baseline = reg.log_model(
        best_model,
        model_name="FRAUD_DETECTION_MODEL",
        version_name="v1_baseline",
        sample_input_data=X_train_basic.iloc[:50],
        task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        metrics=metrics_baseline
    )

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")
print(f"\nTest Set Results:")
for metric, value in metrics_baseline.items():
    print(f"  {metric}: {value:.4f}")

## Experiment 2: Feature Store Model (Enriched Features)

XGBoost classifier using customer and terminal features from the Feature Store.

In [ ]:
spine_df = session.table("TRANSACTIONS_RAW").select(
    "TRANSACTION_ID", "CUSTOMER_ID", "TERMINAL_ID",
    to_timestamp("TX_DATETIME").alias("TX_DATETIME"),
    "TX_AMOUNT", "TX_FRAUD"
).with_columns(
    ['HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT'],
    [
        hour('TX_DATETIME'),
        dayofweek('TX_DATETIME'),
        when(dayofweek('TX_DATETIME') >= 6, True).otherwise(False),
        when(hour('TX_DATETIME').between(0, 6), True).otherwise(False)
    ]
)

training_dataset = fs.generate_dataset(
    name="FRAUD_TRAINING_DATASET",
    spine_df=spine_df,
    features=[registered_customer_fv, registered_terminal_fv],
    version="v1",
    spine_timestamp_col="TX_DATETIME",
    spine_label_cols=["TX_FRAUD"],
    desc="Training dataset with customer and terminal features"
)

training_dataset.read.to_snowpark_dataframe().limit(5).show()

In [ ]:
df_v2 = training_dataset.read.to_snowpark_dataframe().to_pandas()

features_v2 = [
    'TX_AMOUNT', 'HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT',
    'CUST_TX_COUNT_1H', 'CUST_TX_COUNT_24H', 'CUST_TX_COUNT_7D',
    'CUST_AMOUNT_1H', 'CUST_AMOUNT_24H', 'CUST_AVG_AMOUNT_7D', 'CUST_STD_AMOUNT_7D',
    'TERM_TX_COUNT_1H', 'TERM_TX_COUNT_24H', 'TERM_AMOUNT_24H',
    'TERM_AVG_AMOUNT_7D', 'TERM_FRAUD_COUNT_7D'
]

X_v2 = df_v2[features_v2].fillna(0).astype(float)
y_v2 = df_v2['TX_FRAUD'].astype(int)

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)

scale_pos_weight_v2 = len(y_train_v2[y_train_v2 == 0]) / len(y_train_v2[y_train_v2 == 1])

param_grid_v2 = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

model_v2 = XGBClassifier(
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight_v2,
    n_jobs=-1
)

grid_search_v2 = GridSearchCV(
    estimator=model_v2,
    param_grid=param_grid_v2,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

with exp.start_run("feature_store_enriched"):
    exp.log_params({
        "model_type": "XGBClassifier",
        "feature_set": "customer_terminal",
        "n_features": len(features_v2),
        "n_train_rows": len(X_train_v2),
        "n_test_rows": len(X_test_v2),
        "param_grid": str(param_grid_v2),
        "scale_pos_weight": scale_pos_weight_v2,
        "feature_views": "CUSTOMER_FRAUD_FEATURES, TERMINAL_FRAUD_FEATURES"
    })

    grid_search_v2.fit(X_train_v2, y_train_v2)
    best_model_v2 = grid_search_v2.best_estimator_

    exp.log_params({
        "best_n_estimators": grid_search_v2.best_params_['n_estimators'],
        "best_max_depth": grid_search_v2.best_params_['max_depth'],
        "best_learning_rate": grid_search_v2.best_params_['learning_rate'],
        "best_cv_score": grid_search_v2.best_score_
    })

    y_pred_v2 = best_model_v2.predict(X_test_v2)
    y_proba_v2 = best_model_v2.predict_proba(X_test_v2)[:, 1]

    metrics_v2 = {
        "accuracy": float(accuracy_score(y_test_v2, y_pred_v2)),
        "precision": float(precision_score(y_test_v2, y_pred_v2, zero_division=0)),
        "recall": float(recall_score(y_test_v2, y_pred_v2, zero_division=0)),
        "f1_score": float(f1_score(y_test_v2, y_pred_v2, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test_v2, y_proba_v2)),
        "pr_auc": float(average_precision_score(y_test_v2, y_proba_v2))
    }
    exp.log_metrics(metrics_v2)

    mv_v2 = reg.log_model(
        best_model_v2,
        model_name="FRAUD_DETECTION_MODEL",
        version_name="v2_feature_store",
        sample_input_data=X_train_v2.iloc[:50],
        task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        metrics=metrics_v2
    )

print(f"Best params: {grid_search_v2.best_params_}")
print(f"Best CV ROC-AUC: {grid_search_v2.best_score_:.4f}")
print(f"\nTest Set Results:")
for metric, value in metrics_v2.items():
    print(f"  {metric}: {value:.4f}")

## Model Comparison

Compare baseline and feature-enriched models across all metrics.

In [ ]:
comparison_data = {
    'Version': ['v1_baseline', 'v2_feature_store'],
    'Features': ['Basic (5)', 'Customer+Terminal (17)'],
    'Accuracy': [metrics_baseline['accuracy'], metrics_v2['accuracy']],
    'Precision': [metrics_baseline['precision'], metrics_v2['precision']],
    'Recall': [metrics_baseline['recall'], metrics_v2['recall']],
    'F1 Score': [metrics_baseline['f1_score'], metrics_v2['f1_score']],
    'ROC AUC': [metrics_baseline['roc_auc'], metrics_v2['roc_auc']],
    'PR AUC': [metrics_baseline['pr_auc'], metrics_v2['pr_auc']]
}

comparison_df = pd.DataFrame(comparison_data)
print("=" * 90)
print("MODEL COMPARISON")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("=" * 90)

print("\nImprovement (v2 over v1):")
print("-" * 50)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'PR AUC']:
    v1 = comparison_df.loc[0, metric]
    v2 = comparison_df.loc[1, metric]
    if v1 > 0:
        pct = ((v2 - v1) / v1) * 100
        print(f"  {metric:12s}: {v1:.4f} -> {v2:.4f} ({pct:+.2f}%)")
    else:
        print(f"  {metric:12s}: {v1:.4f} -> {v2:.4f}")

In [ ]:
all_metrics = {
    'v1_baseline': metrics_baseline,
    'v2_feature_store': metrics_v2
}

best_version = max(all_metrics.keys(), key=lambda v: all_metrics[v]['f1_score'])
best_f1 = all_metrics[best_version]['f1_score']

model = reg.get_model("FRAUD_DETECTION_MODEL")
model.default = best_version

print(f"Best model by F1 score: {best_version} (F1={best_f1:.4f})")
print(f"Default version set to: {best_version}")
print("\nAll registered versions:")
model.show_versions()

## Summary

This notebook built an end-to-end fraud detection pipeline using Snowflake ML:

1. **Data Ingestion** — Loaded 1.75M transactions into Snowflake
2. **Feature Store** — Registered customer and terminal entities with rolling-window feature views
3. **Experiment 1** — Baseline XGBoost with 5 basic features
4. **Experiment 2** — Enriched XGBoost with 17 features from the Feature Store
5. **Model Registry** — Logged both versions and promoted the best by F1 score

**Next steps:**
- View experiment results in Snowsight under AI & ML > Experiments
- Inspect model versions in AI & ML > Model Registry
- Enable ML Observability to monitor model drift in production